In [1]:
import pandas as pd
from pathlib import Path

relative_path = Path("checkpoints/note01_initial.pkl")

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / relative_path).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Không tìm thấy file: {relative_path}"
    )
df_path = project_root / relative_path

df = pd.read_pickle(df_path)

print("Loaded checkpoint:", df_path)
print("Dataset shape:", df.shape)

Loaded checkpoint: d:\phan tich data\Ember_Electricity\checkpoints\note01_initial.pkl
Dataset shape: (104203, 24)


In [2]:
# Kiểm tra kiểu dữ liệu và số lượng giá trị thiếu
structure_check = pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique()
})

structure_check

,dtype,missing,missing_pct,n_unique
Area,str,0,0.00,224
ISO 3 code,str,6266,6.01,209
Year,int64,0,0.00,41
Area type,str,0,0.00,2
Electricity source,str,0,0.00,17
Is aggregated source,bool,0,0.00,2
Generation (TWh),float64,0,0.00,17453
Generation YoY change (TWh),float64,3744,3.59,8386
Generation YoY change (%),float64,37203,35.70,25779
Share of generation (%),float64,34,0.03,30078


In [3]:
missing_by_area_type = (
    df.groupby("Area type")
      .agg(
          rows=("Area", "size"),
          missing_iso=("ISO 3 code", lambda x: x.isna().sum()),
          missing_continent=("Continent", lambda x: x.isna().sum()),
          missing_region=("Ember region", lambda x: x.isna().sum()),
          missing_capacity=("Capacity (GW)", lambda x: x.isna().sum()),
          missing_emissions=("Emissions (MtCO2e)", lambda x: x.isna().sum()),
          missing_intensity=("Emissions intensity (gCO2e/kWh)", lambda x: x.isna().sum())
      )
)

missing_by_area_type

,rows,missing_iso,missing_continent,missing_region,missing_capacity,missing_emissions,missing_intensity
Area type,,,,,,,
Country or economy,97937,0,0,0,27110,5915,40207
Region,6266,6266,6266,6266,1196,390,543


In [4]:
missing_by_source = (
    df.groupby("Electricity source")
      .agg(
          rows=("Area", "size"),
          missing_capacity=("Capacity (GW)", lambda x: x.isna().sum()),
          missing_emissions=("Emissions (MtCO2e)", lambda x: x.isna().sum()),
          missing_intensity=("Emissions intensity (gCO2e/kWh)", lambda x: x.isna().sum())
      )
)

missing_by_source

,rows,missing_capacity,missing_emissions,missing_intensity
Electricity source,,,,
Bioenergy,6233,735,0,2841
Clean,6298,766,0,926
Coal,6189,3639,0,3151
Demand,6305,6305,6305,6305
Fossil,6305,719,0,186
Gas,6122,2587,0,2351
Hydro,6190,747,0,1346
"Hydro, bioenergy and other renewables",6298,763,0,1258
Net imports,5632,5632,0,2236


In [5]:
source_coverage = (
    df.groupby("Electricity source")
      .agg(
          first_year=("Year", "min"),
          last_year=("Year", "max"),
          years=("Year", "nunique"),
          areas=("Area", "nunique")
      )
      .sort_values("first_year")
)

source_coverage

,first_year,last_year,years,areas
Electricity source,,,,
Bioenergy,1985,2025,41,224
Clean,1985,2025,41,224
Coal,1985,2025,41,220
Demand,1985,2025,41,224
Fossil,1985,2025,41,224
Gas,1985,2025,41,217
Hydro,1985,2025,41,222
"Hydro, bioenergy and other renewables",1985,2025,41,224
Nuclear,1985,2025,41,215


In [6]:
country_df = df[df["Area type"] == "Country or economy"].copy()

country_coverage = (
    country_df.groupby("Area")["Year"]
             .agg(
                 first_year="min",
                 last_year="max",
                 years="nunique"
             )
             .sort_values("years")
)

country_coverage.describe()

,first_year,last_year,years
count,209.000000,209.000000,209.000000
mean,1997.066986,2024.368421,28.301435
std,5.512687,0.630291,5.822705
min,1985.000000,2022.000000,13.000000
25%,2000.000000,2024.000000,25.000000
50%,2000.000000,2024.000000,25.000000
75%,2000.000000,2025.000000,26.000000
max,2012.000000,2025.000000,41.000000


In [7]:
core_coverage = (
    country_df.groupby("Year")
              .agg(
                  countries=("Area", "nunique"),
                  generation=("Generation (TWh)", lambda x: x.notna().sum()),
                  renewable_share=("Share of generation (%)", lambda x: x.notna().sum()),
                  emissions=("Emissions (MtCO2e)", lambda x: x.notna().sum())
              )
)

core_coverage

,countries,generation,renewable_share,emissions
Year,,,,
1985,22,289,289,267
1986,22,289,289,267
1987,22,289,289,267
1988,22,289,289,267
1989,22,289,289,267
1990,52,843,843,791
1991,52,843,843,791
1992,52,843,843,791
1993,52,843,843,791


In [8]:
numeric_cols = [
    "Generation (TWh)",
    "Generation YoY change (TWh)",
    "Generation YoY change (%)",
    "Share of generation (%)",
    "Share of generation YoY change (% points)",
    "Capacity (GW)",
    "Emissions (MtCO2e)",
    "Emissions YoY change (MtCO2e)",
    "Emissions YoY change (%)",
    "Share of emissions (%)",
    "Emissions intensity (gCO2e/kWh)"
]

negative_check = (
    df[numeric_cols]
    .lt(0)
    .sum()
    .sort_values(ascending=False)
)

negative_check

Share of generation YoY change (% points)    25766
Generation YoY change (%)                    19635
Generation YoY change (TWh)                  19576
Emissions YoY change (MtCO2e)                16149
Emissions YoY change (%)                     16147
Generation (TWh)                              1409
Share of generation (%)                       1409
Share of emissions (%)                          17
Emissions (MtCO2e)                              11
Emissions intensity (gCO2e/kWh)                  1
Capacity (GW)                                    0
dtype: int64

In [9]:
negative_generation = (
    df[df["Generation (TWh)"] < 0]
    .groupby("Electricity source")
    .size()
    .sort_values(ascending=False)
)

negative_generation

Electricity source
Net imports         1397
Other fossil           5
Other renewables       4
Fossil                 1
Solar                  1
Wind                   1
dtype: int64

In [10]:
negative_generation_details = (
    df[
        (df["Generation (TWh)"] < 0)
        & (df["Electricity source"] != "Net imports")
    ][
        [
            "Area",
            "ISO 3 code",
            "Year",
            "Area type",
            "Electricity source",
            "Generation (TWh)",
            "Share of generation (%)",
            "Emissions (MtCO2e)"
        ]
    ]
    .sort_values(["Electricity source", "Area", "Year"])
)

negative_generation_details

,Area,ISO 3 code,Year,Area type,Electricity source,Generation (TWh),Share of generation (%),Emissions (MtCO2e)
21899,Costa Rica,CRI,2025,Country or economy,Fossil,-1.150,-9.863,-0.757
10128,Belgium,BEL,2025,Country or economy,Other fossil,-0.349,-0.485,-0.230
21893,Costa Rica,CRI,2025,Country or economy,Other fossil,-1.150,-9.863,-0.757
22926,Croatia,HRV,2025,Country or economy,Other fossil,-0.003,-0.020,-0.002
51677,Kyrgyzstan,KGZ,2025,Country or economy,Other fossil,-0.144,-1.135,-0.095
85474,Slovenia,SVN,2025,Country or economy,Other fossil,-0.019,-0.127,-0.012
24566,Czechia,CZE,2025,Country or economy,Other renewables,-0.111,-0.147,-0.004
29746,Estonia,EST,2025,Country or economy,Other renewables,-0.001,-0.016,0.000
66822,Netherlands,NLD,2025,Country or economy,Other renewables,-0.005,-0.004,0.000
88210,Spain,ESP,2025,Country or economy,Other renewables,-0.045,-0.016,-0.002


In [11]:
share_generation_invalid = (
    df[
        (df["Share of generation (%)"] < 0)
        | (df["Share of generation (%)"] > 100)
    ][
        [
            "Area",
            "Year",
            "Electricity source",
            "Generation (TWh)",
            "Share of generation (%)"
        ]
    ]
    .sort_values(["Year", "Area", "Electricity source"])
)

print("Số dòng Share of generation (%) ngoài [0, 100]:",
      len(share_generation_invalid))

share_generation_invalid

Số dòng Share of generation (%) ngoài [0, 100]: 3689


,Area,Year,Electricity source,Generation (TWh),Share of generation (%)
6321,Austria,1990,Net imports,-0.460,-0.934
9534,Belgium,1990,Net imports,-3.720,-5.292
13729,Bulgaria,1990,Demand,45.690,109.045
22343,Croatia,1990,Demand,15.690,178.093
23977,Czechia,1990,Net imports,-0.690,-1.108
...,...,...,...,...,...
98920,United Kingdom,2025,Demand,311.308,106.110
99362,United States,2025,Demand,4532.142,100.273
99804,Uruguay,2025,Demand,14.783,123.160
100246,Uzbekistan,2025,Demand,89.630,103.379


In [12]:
generation_sources = [
    "Solar",
    "Wind",
    "Hydro",
    "Bioenergy",
    "Other renewables",
    "Nuclear",
    "Coal",
    "Gas",
    "Other fossil"
]

source_share_invalid = (
    df[
        df["Electricity source"].isin(generation_sources)
        & (
            (df["Share of generation (%)"] < 0)
            | (df["Share of generation (%)"] > 100)
        )
    ][
        [
            "Area",
            "Year",
            "Electricity source",
            "Generation (TWh)",
            "Share of generation (%)"
        ]
    ]
    .sort_values(["Year", "Area", "Electricity source"])
)

print(
    "Số dòng nguồn phát điện có Share of generation ngoài [0, 100]:",
    len(source_share_invalid)
)

source_share_invalid

Số dòng nguồn phát điện có Share of generation ngoài [0, 100]: 11


,Area,Year,Electricity source,Generation (TWh),Share of generation (%)
7349,Azerbaijan,2025,Solar,-0.094,-0.340
8642,Bangladesh,2025,Wind,-0.038,-0.037
10128,Belgium,2025,Other fossil,-0.349,-0.485
21893,Costa Rica,2025,Other fossil,-1.150,-9.863
22926,Croatia,2025,Other fossil,-0.003,-0.020
24566,Czechia,2025,Other renewables,-0.111,-0.147
29746,Estonia,2025,Other renewables,-0.001,-0.016
51677,Kyrgyzstan,2025,Other fossil,-0.144,-1.135
66822,Netherlands,2025,Other renewables,-0.005,-0.004
85474,Slovenia,2025,Other fossil,-0.019,-0.127


In [13]:
share_emissions_invalid = (
    df[
        (df["Share of emissions (%)"] < 0)
        | (df["Share of emissions (%)"] > 100)
    ][
        [
            "Area",
            "Year",
            "Electricity source",
            "Emissions (MtCO2e)",
            "Share of emissions (%)"
        ]
    ]
    .sort_values(["Year", "Area", "Electricity source"])
)

print(
    "Số dòng Share of emissions (%) ngoài [0, 100]:",
    len(share_emissions_invalid)
)

share_emissions_invalid

Số dòng Share of emissions (%) ngoài [0, 100]: 3597


,Area,Year,Electricity source,Emissions (MtCO2e),Share of emissions (%)
3808,Argentina,1985,Total generation,11.766,100.003
12664,Brazil,1985,Total generation,11.797,100.002
57763,Malaysia,1985,Total generation,7.386,100.006
60958,Mexico,1985,Total generation,45.027,100.001
81499,Saudi Arabia,1985,Fossil,36.206,100.001
...,...,...,...,...,...
77927,Romania,2025,Total generation,12.929,100.001
85474,Slovenia,2025,Other fossil,-0.012,-0.426
88210,Spain,2025,Other renewables,-0.002,-0.005
95162,Tunisia,2025,Total generation,12.504,100.002


In [14]:
renewable_components = [
    "Solar",
    "Wind",
    "Hydro",
    "Bioenergy",
    "Other renewables"
]

renewable_check = (
    df[df["Electricity source"].isin(renewable_components)]
    .groupby(["Area", "Year"])["Generation (TWh)"]
    .sum()
    .rename("component_renewables")
    .reset_index()
)

renewable_aggregate = (
    df[df["Electricity source"] == "Renewables"]
    [["Area", "Year", "Generation (TWh)"]]
    .rename(columns={"Generation (TWh)": "aggregate_renewables"})
)

renewable_check = renewable_check.merge(
    renewable_aggregate,
    on=["Area", "Year"],
    how="inner"
)

renewable_check["difference"] = (
    renewable_check["component_renewables"]
    - renewable_check["aggregate_renewables"]
)

print("Số cặp Area-Year được kiểm tra:", len(renewable_check))
print("\nSai lệch lớn nhất:")
print(renewable_check["difference"].abs().max())

renewable_check.sort_values(
    "difference",
    key=lambda x: x.abs(),
    ascending=False
).head(10)

Số cặp Area-Year được kiểm tra: 6298

Sai lệch lớn nhất:
0.0020000000004074536


,Area,Year,component_renewables,aggregate_renewables,difference
2095,G20,2025,9093.462,9093.464,-0.002
332,Asia,2012,1565.591,1565.589,0.002
5161,Slovenia,2025,5.878,5.880,-0.002
6209,World,2012,4729.821,4729.819,0.002
340,Asia,2020,3097.907,3097.906,0.001
344,Asia,2024,4609.035,4609.034,0.001
1162,China,2014,1276.109,1276.108,0.001
1168,China,2020,2139.829,2139.828,0.001
345,Asia,2025,5280.721,5280.722,-0.001
6213,World,2016,5859.464,5859.465,-0.001


In [15]:
fossil_components = [
    "Coal",
    "Gas",
    "Other fossil"
]

fossil_check = (
    df[df["Electricity source"].isin(fossil_components)]
    .groupby(["Area", "Year"])["Generation (TWh)"]
    .sum()
    .rename("component_fossil")
    .reset_index()
)

fossil_aggregate = (
    df[df["Electricity source"] == "Fossil"]
    [["Area", "Year", "Generation (TWh)"]]
    .rename(columns={"Generation (TWh)": "aggregate_fossil"})
)

fossil_check = fossil_check.merge(
    fossil_aggregate,
    on=["Area", "Year"],
    how="inner"
)

fossil_check["difference"] = (
    fossil_check["component_fossil"]
    - fossil_check["aggregate_fossil"]
)

print("Số cặp Area-Year được kiểm tra:", len(fossil_check))
print("Sai lệch lớn nhất:",
      fossil_check["difference"].abs().max())

fossil_check.sort_values(
    "difference",
    key=lambda x: x.abs(),
    ascending=False
).head(10)

Số cặp Area-Year được kiểm tra: 6305
Sai lệch lớn nhất: 0.0010000000002037268


,Area,Year,component_fossil,aggregate_fossil,difference
341,Asia,2021,9860.385,9860.384,0.001
2079,G20,2009,11323.816,11323.815,0.001
326,Asia,2006,4952.651,4952.650,0.001
3412,MENA,2025,1851.032,1851.031,0.001
2073,G20,2003,9522.907,9522.908,-0.001
2091,G20,2021,14564.539,14564.538,0.001
6225,World,2021,17653.955,17653.954,0.001
6206,World,2002,10560.975,10560.976,-0.001
1168,China,2020,5239.531,5239.532,-0.001
1166,China,2018,5031.213,5031.212,0.001


In [16]:
id="m3k7qp"
generation_check = (
    df[df["Electricity source"].isin(generation_sources)]
    .groupby(["Area", "Year"])["Generation (TWh)"]
    .sum()
    .rename("component_generation")
    .reset_index()
)

total_generation = (
    df[df["Electricity source"] == "Total generation"]
    [["Area", "Year", "Generation (TWh)"]]
    .rename(columns={"Generation (TWh)": "total_generation"})
)

generation_check = generation_check.merge(
    total_generation,
    on=["Area", "Year"],
    how="inner"
)

generation_check["difference"] = (
    generation_check["component_generation"]
    - generation_check["total_generation"]
)

print("Số cặp Area-Year được kiểm tra:", len(generation_check))
print(
    "Sai lệch lớn nhất:",
    generation_check["difference"].abs().max()
)

generation_check.sort_values(
    "difference",
    key=lambda x: x.abs(),
    ascending=False
).head(10)

Số cặp Area-Year được kiểm tra: 6305
Sai lệch lớn nhất: 0.0020000000004074536


,Area,Year,component_generation,total_generation,difference
6206,World,2002,16098.027,16098.029,-0.002
322,Asia,2002,4722.307,4722.309,-0.002
2083,G20,2013,19550.349,19550.351,-0.002
344,Asia,2024,16181.149,16181.147,0.002
10,ASEAN,2010,744.599,744.597,0.002
2687,Indonesia,2010,235.294,235.292,0.002
4290,Norway,2025,160.853,160.855,-0.002
575,Belarus,2025,45.431,45.429,0.002
2075,G20,2005,15407.155,15407.156,-0.001
1162,China,2014,5781.268,5781.267,0.001


In [17]:
system_sources = ["Demand", "Net imports"]

system_source_summary = (
    df[df["Electricity source"].isin(system_sources)]
    .groupby("Electricity source")
    .agg(
        rows=("Area", "size"),
        areas=("Area", "nunique"),
        years=("Year", "nunique"),
        min_generation=("Generation (TWh)", "min"),
        max_generation=("Generation (TWh)", "max")
    )
)

system_source_summary["missing_generation"] = (
    df[df["Electricity source"].isin(system_sources)]
    .groupby("Electricity source")["Generation (TWh)"]
    .apply(lambda x: x.isna().sum())
)

system_source_summary

,rows,areas,years,min_generation,max_generation,missing_generation
Electricity source,,,,,,
Demand,6305,224,41,0.01,31820.135,0
Net imports,5632,210,36,-93.80,66.670,0


In [18]:
note02_data_quality = (
    project_root / "checkpoints/note02_data_quality.pkl"
)

note02_state = {
    "df": df,
    "generation_sources": generation_sources,
    "system_sources": system_sources,
    "country_df": country_df,
    "country_coverage": country_coverage,
    "core_coverage": core_coverage,
    "renewable_check": renewable_check,
    "fossil_check": fossil_check,
    "generation_check": generation_check,
    "system_source_summary": system_source_summary
}

pd.to_pickle(note02_state, note02_data_quality)

print("Saved Note 2 checkpoint:")
print(note02_data_quality)

Saved Note 2 checkpoint:
d:\phan tich data\Ember_Electricity\checkpoints\note02_data_quality.pkl
